# Artificial Intelligence — Lab 6
## Simulated Annealing and Optimization Performance

**Course Learning Outcome — CLO3**  
Illustrate and analyze the performance of local, global, and optimal search methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experiments, justifications, debugging answers, and reflection.

> **Assessment principle:** Correct code is only one part of the evidence. Most marks come from your ability to **reason about temperature, acceptance probability, cooling schedules, stochastic behavior, and the trade-off between exploration and exploitation**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Simulated annealing concepts | 15 min | Connect temperature to exploration |
| 2. Acceptance-probability reasoning | 20 min | Predict whether worse moves may be accepted |
| 3. Implement simulated annealing | 30 min | Build a stochastic local-search algorithm |
| 4. Compare cooling schedules | 20 min | Analyze solution quality and search effort |
| 5. Compare with hill climbing | 20 min | Measure robustness across repeated runs |
| 6. Debugging, variation & reflection | 15 min | Diagnose errors and defend conclusions |

> **Main idea:** Unlike hill climbing, simulated annealing may intentionally accept a worse move, especially at high temperature, to escape local optima.

## Learning Objectives

By the end of this lab, you should be able to:

1. explain the purpose of simulated annealing;
2. distinguish improving and worsening moves;
3. compute an acceptance probability for a worse move;
4. explain the role of temperature $T$;
5. implement simulated annealing;
6. compare different cooling schedules;
7. analyze repeated stochastic runs using success rate and average solution quality;
8. compare simulated annealing with hill climbing;
9. diagnose common implementation mistakes;
10. justify parameter choices for a local optimization problem.

In [ ]:
import math
import random
import statistics
import matplotlib.pyplot as plt

print("Lab 6 environment ready.")

# Part I — Optimization Landscape

We use the same rugged landscape as the previous local-search lab:

$$
f(x)=8\sin\left(\frac{x}{6}\right)+5\sin\left(\frac{x}{2.7}\right)+0.08x
$$

for integer states $x\in\{0,1,\ldots,100\}$. The objective is to **maximize** $f(x)$.

In [ ]:
DOMAIN_MIN = 0
DOMAIN_MAX = 100

def objective(x: int) -> float:
    return 8*math.sin(x/6) + 5*math.sin(x/2.7) + 0.08*x

xs = list(range(DOMAIN_MIN, DOMAIN_MAX + 1))
ys = [objective(x) for x in xs]
best_global_x = max(xs, key=objective)
best_global_value = objective(best_global_x)

plt.figure(figsize=(10,4))
plt.plot(xs, ys)
plt.xlabel("State x")
plt.ylabel("Objective f(x)")
plt.title("Optimization Landscape")
plt.grid(True, alpha=0.25)
plt.show()

print("Global optimum state:", best_global_x)
print("Global optimum value:", round(best_global_value, 4))

## Task 1.1 — Recall Hill Climbing

1. Why can hill climbing fail on this landscape?
2. What happens when every immediate neighbor has a lower value?
3. What feature of simulated annealing is intended to help with this problem?

**Your answers:**

# Part II — Acceptance Probability

For a maximization problem, define

$$
\Delta=f(s')-f(s).
$$

If $\Delta>0$, accept the new state. If $\Delta\le0$, simulated annealing may still accept it with probability

$$
P(\text{accept})=e^{\Delta/T}.
$$

Because $\Delta$ is negative for a worse move, the probability lies between 0 and 1.

## Task 2.1 — Manual Probability Calculations

Compute the probabilities before running the code.

| Case | $\Delta$ | $T$ | $e^{\Delta/T}$ | Interpretation |
|---|---:|---:|---:|---|
| A | -1 | 10 |  |  |
| B | -1 | 1 |  |  |
| C | -5 | 10 |  |  |
| D | -5 | 1 |  |  |

Then answer:

1. What happens to acceptance probability as $T$ decreases?
2. What happens when a move becomes much worse?
3. Why does high temperature encourage **exploration**?

**Your answers:**

In [ ]:
examples = [("A",-1,10),("B",-1,1),("C",-5,10),("D",-5,1)]
for name, delta, T in examples:
    print(name, "->", round(math.exp(delta/T), 4))

## Task 2.2 — Predict Acceptance from a Random Draw

Suppose $P=0.30$ and a move is accepted when $r<P$.

| Random value $r$ | Accept or reject? | Why? |
|---:|---|---|
| 0.12 |  |  |
| 0.29 |  |  |
| 0.31 |  |  |
| 0.85 |  |  |

# Part III — Neighborhood and Cooling

Use the neighborhood

$$
N(x)=\{x-1,x+1\}
$$

when states remain inside the domain.

A geometric cooling schedule is

$$
T_{k+1}=\alpha T_k,
$$

where $0<\alpha<1$.

In [ ]:
def neighbors(x: int):
    result=[]
    if x>DOMAIN_MIN: result.append(x-1)
    if x<DOMAIN_MAX: result.append(x+1)
    return result

## Task 3.1 — Cooling Prediction

Assume $T_0=10$.

Compute the temperature after three updates for:

- $\alpha=0.95$
- $\alpha=0.80$

Then answer:

1. Which schedule remains exploratory longer?
2. Which reaches low temperature faster?
3. What is the likely trade-off between slow and fast cooling?

**Your answers:**

# Part IV — Implement Simulated Annealing

Your function should return

```python
(best_state, best_value, trace)
```

The trace records iteration, current state, candidate, temperature, $\Delta$, and whether the move was accepted.

In [ ]:
def simulated_annealing(start, initial_temperature=10.0, cooling_rate=0.95,
                        min_temperature=0.01, max_iterations=1000, rng=None):
    if rng is None:
        rng = random.Random()

    current = start
    current_value = objective(current)
    best_state, best_value = current, current_value
    temperature = initial_temperature
    trace = []

    for iteration in range(max_iterations):
        if temperature < min_temperature:
            break

        # TODO 1: randomly choose one legal neighbor
        candidate = None

        # TODO 2: compute candidate value and delta
        candidate_value = None
        delta = None

        # TODO 3: accept improving moves; otherwise accept with exp(delta/T)
        accepted = False

        # TODO 4: if accepted, update current/current_value

        # TODO 5: update best_state/best_value if necessary

        trace.append({
            "iteration": iteration,
            "current": current,
            "current_value": current_value,
            "temperature": temperature,
            "candidate": candidate,
            "candidate_value": candidate_value,
            "delta": delta,
            "accepted": accepted,
        })

        # TODO 6: cool the temperature

    return best_state, best_value, trace

## Task 4.1 — Predict Before Running

Use `start = 10`, $T_0=10$, $\alpha=0.95$, and random seed 42.

Before running, answer:

1. Do you expect every accepted move to improve the objective?
2. Do you expect the final current state and best-ever state always to be identical?
3. Why should the algorithm store the best state seen?

**Your prediction:**

In [ ]:
rng = random.Random(42)
best_state, best_value, trace = simulated_annealing(
    start=10, initial_temperature=10.0, cooling_rate=0.95, rng=rng
)

print("Best state:", best_state)
print("Best objective:", round(best_value, 4))
print("Iterations:", len(trace))
print("Global optimum:", best_global_x, round(best_global_value, 4))

## Task 4.2 — Inspect Early Decisions

Print the first 12 iterations and identify:

- one improving move;
- one accepted worse move, if present;
- one rejected worse move, if present.

Then explain why each decision is consistent with the algorithm.

In [ ]:
for row in trace[:12]:
    print(
        f"iter={row['iteration']:>3} "
        f"T={row['temperature']:.3f} "
        f"candidate={row['candidate']} "
        f"delta={row['delta']} "
        f"accepted={row['accepted']} "
        f"current={row['current']}"
    )

## Task 4.3 — Explain the Trace

1. Why are improving moves always accepted?
2. Why can a worse move be accepted early in the run?
3. Why does the same worsening move become less likely later?
4. Why is simulated annealing more exploratory early and more conservative later?

**Your answers:**

# Part V — Visualize Search Behavior

In [ ]:
visited_values = [row["current_value"] for row in trace]
plt.figure(figsize=(10,4))
plt.plot(range(len(visited_values)), visited_values)
plt.xlabel("Iteration")
plt.ylabel("Current objective")
plt.title("Simulated Annealing: Objective over Time")
plt.grid(True, alpha=0.25)
plt.show()

## Task 5.1 — Interpret the Plot

1. Does the objective improve monotonically?
2. Why is non-monotonic behavior expected?
3. What can a temporary decrease help the algorithm achieve?
4. How would this differ from strict hill climbing?

**Your answers:**

# Part VI — Compare Cooling Schedules

Compare:

- fast cooling: $\alpha=0.80$
- medium cooling: $\alpha=0.95$
- slow cooling: $\alpha=0.99$

Use repeated trials rather than a single run.

In [ ]:
def run_sa_trials(cooling_rate, trials=100, initial_temperature=10.0, seed=123):
    master_rng = random.Random(seed)
    values, lengths = [], []
    successes = 0

    for _ in range(trials):
        rng = random.Random(master_rng.randint(0, 10**9))
        start = rng.randint(DOMAIN_MIN, DOMAIN_MAX)
        _, value, tr = simulated_annealing(
            start=start, initial_temperature=initial_temperature,
            cooling_rate=cooling_rate, rng=rng
        )
        values.append(value)
        lengths.append(len(tr))
        if abs(value - best_global_value) <= 1e-9:
            successes += 1

    return {
        "cooling_rate": cooling_rate,
        "success_rate": successes/trials,
        "mean_final_value": statistics.mean(values),
        "best_value": max(values),
        "worst_value": min(values),
        "mean_iterations": statistics.mean(lengths),
    }

## Task 6.1 — Predict Before Running

1. Which cooling rate do you expect to have the highest success rate?
2. Which should use the most iterations?
3. Why might extremely slow cooling be expensive?
4. Why might extremely fast cooling behave too much like hill climbing?

**Your prediction:**

In [ ]:
schedule_results = [run_sa_trials(a) for a in [0.80, 0.95, 0.99]]
for row in schedule_results:
    print(row)

## Task 6.2 — Analyze Cooling Performance

| Cooling rate | Success rate | Mean final objective | Mean iterations |
|---:|---:|---:|---:|
| 0.80 |  |  |  |
| 0.95 |  |  |  |
| 0.99 |  |  |  |

Then answer:

1. Which setting performed best by success rate?
2. Which required the most iterations?
3. What trade-off is visible?
4. Why should a cooling rate not be chosen from one run only?

**Your answers:**

# Part VII — Compare with Hill Climbing

In [ ]:
def hill_climb(start):
    current=start
    while True:
        candidates=neighbors(current)
        best=max(candidates, key=objective)
        if objective(best) <= objective(current):
            return current, objective(current)
        current=best

def compare_hc_sa(trials=200, seed=77):
    master=random.Random(seed)
    hc_values, sa_values=[], []
    hc_success=sa_success=0

    for _ in range(trials):
        start=master.randint(DOMAIN_MIN, DOMAIN_MAX)
        _, hv=hill_climb(start)
        hc_values.append(hv)
        hc_success += abs(hv-best_global_value) <= 1e-9

        rng=random.Random(master.randint(0,10**9))
        _, sv, _=simulated_annealing(start, 10.0, 0.95, rng=rng)
        sa_values.append(sv)
        sa_success += abs(sv-best_global_value) <= 1e-9

    return {
        "hill_climbing_success_rate": hc_success/trials,
        "simulated_annealing_success_rate": sa_success/trials,
        "hill_climbing_mean_value": statistics.mean(hc_values),
        "simulated_annealing_mean_value": statistics.mean(sa_values),
    }

comparison=compare_hc_sa()
comparison

## Task 7.1 — Interpret Hill Climbing vs. Simulated Annealing

| Metric | Hill Climbing | Simulated Annealing |
|---|---:|---:|
| Success rate |  |  |
| Mean final objective |  |  |

Then answer:

1. Which reached the global optimum more often?
2. Which produced the better average solution?
3. Why can simulated annealing outperform hill climbing on rugged landscapes?
4. Why can hill climbing still be attractive?
5. Which method is deterministic here?

**Your answers:**

# Part VIII — Debugging Simulated Annealing

## Task 8.1 — Wrong Sign

A student uses:

```python
p = math.exp(-delta / temperature)
```

where `delta = candidate_value - current_value`.

1. Why can this produce a probability greater than 1 for a worse move?
2. What should the formula be?
3. Why is sign handling important?

**Your answer:**

## Task 8.2 — Never Accept Worse Moves

A student writes:

```python
if delta > 0:
    current = candidate
```

1. Which algorithm does this resemble?
2. What simulated-annealing capability is lost?
3. Why can this cause local-optimum trapping?

**Your answer:**

## Task 8.3 — No Cooling

A student keeps `temperature = 10` for the entire run.

1. What happens to exploration over time?
2. Why may poor moves continue to be accepted too often?
3. What role does cooling play in convergence?

**Your answer:**

# Part IX — Personalized Parameter Variation

Use the last digit of your student ID:

- `0–3`: initial temperature $T_0=2$
- `4–6`: initial temperature $T_0=10$
- `7–9`: initial temperature $T_0=30$

Keep $\alpha=0.95$.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_T0 = 2.0
    elif 4 <= LAST_DIGIT <= 6:
        PERSONAL_T0 = 10.0
    elif 7 <= LAST_DIGIT <= 9:
        PERSONAL_T0 = 30.0
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")
    print("Assigned initial temperature:", PERSONAL_T0)

## Task 9.1 — Predict Before Running

- **Assigned $T_0$:**
- **Expected early exploration:** more / less
- **Expected frequency of accepted worse moves:**
- **Potential benefit:**
- **Potential drawback:**

**Your prediction:**

In [ ]:
if LAST_DIGIT is not None:
    personal_results = run_sa_trials(
        cooling_rate=0.95, trials=100,
        initial_temperature=PERSONAL_T0, seed=999
    )
    print(personal_results)

## Task 9.2 — Explain the Personalized Result

1. What success rate did your setting achieve?
2. How does its mean objective compare with the earlier runs?
3. Did your exploration prediction appear reasonable?
4. Why can a very low $T_0$ behave almost greedily?
5. Why can a very high $T_0$ waste effort?

**Your answers:**

# Part X — Individual Understanding Check

Your instructor may ask one short question about your notebook, for example:

- Show where a worse move can be accepted.
- Explain the meaning of $\Delta$.
- Why does high temperature increase exploration?
- Why does the algorithm cool over time?
- Why store the best state seen?
- What happens if cooling is too fast?
- Why are repeated trials necessary?
- How does simulated annealing differ from hill climbing?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

### R1 — Exploration vs. Exploitation
How does simulated annealing transition from exploration to exploitation?

**Answer:**

### R2 — Worse Moves
Why can accepting a worse move be rational in optimization?

**Answer:**

### R3 — Cooling
What happens if cooling is too fast? What happens if it is too slow?

**Answer:**

### R4 — Stochastic Evaluation
Why should success rate and mean objective be measured over many runs?

**Answer:**

### R5 — Method Selection
Give one situation where hill climbing may be preferable and one where simulated annealing may be preferable.

**Answer:**

# Submission Checklist

- [ ] acceptance-probability calculations completed;
- [ ] cooling-schedule reasoning completed;
- [ ] simulated annealing implemented correctly;
- [ ] early stochastic decisions interpreted;
- [ ] trace plot analyzed;
- [ ] cooling rates compared;
- [ ] repeated-trial performance analyzed;
- [ ] hill climbing vs. simulated annealing compared;
- [ ] debugging questions answered;
- [ ] personalized temperature experiment completed;
- [ ] prediction written before the personalized run;
- [ ] reflection answers completed;
- [ ] important outputs visible.

Suggested filename:

```text
Lab06_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Simulated annealing and experiments work correctly |
| **Algorithmic justification** | **3.0** | Explains $\Delta$, temperature, acceptance probability, cooling, and exploration/exploitation |
| **Experimental analysis** | **2.0** | Interprets cooling-schedule and hill-climbing comparisons |
| **Trace / prediction / debugging** | **1.0** | Manual probability reasoning, predictions, and diagnosis of faulty logic |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- Simulated annealing may accept improving **and** worsening moves.
- For a worse move,

$$
P(\text{accept})=e^{\Delta/T}.
$$

- High temperature encourages exploration.
- Cooling gradually increases selectivity.
- Slow cooling may improve robustness but costs more computation.
- Repeated trials are essential for evaluating stochastic algorithms.
- Simulated annealing can escape local optima that trap hill climbing.

The next lab will move to **Constraint Satisfaction Problems and Backtracking Search**.